# Python Data Type Internals in CPython

This notebook explains an important mental model:

**Python values may look like primitives, but in CPython they are objects.**

CPython is the standard and most commonly used implementation of Python. It is written mostly in C. Internally, values such as integers, floats, strings, lists, and dictionaries are represented using C structs.

We will not become C extension developers here. The goal is to understand enough internals to write Python with the right mental model.

## 1. Important Correction: `PyInt` vs `PyLong`

In Python 2, there were separate integer-related types internally, including `PyIntObject` and `PyLongObject`.

In Python 3, the normal `int` type is backed by `PyLongObject`. There is no separate public Python-level `int` vs `long` distinction anymore.

So the modern CPython mental model is:

- Python `int` -> CPython `PyLongObject`
- Python `float` -> CPython `PyFloatObject`
- Python `str` -> CPython Unicode object
- Python `list` -> CPython list object
- Python `dict` -> CPython dictionary object

In [ ]:

# Python 3 has only int at the Python language level.
# Very large integers still use the same int type.

small_number = 10
huge_number = 10 ** 100

print(type(small_number))
print(type(huge_number))
print(huge_number)

## 2. Python Object Mental Model

In CPython, every object has some common internal information. Conceptually, an object contains:

- Reference count: how many references point to this object
- Type pointer: what type this object belongs to
- Type-specific data: the actual value or structure

This is very different from a raw C `int`, which is usually just a fixed-size binary value stored directly in memory.

In [ ]:

# id() returns an identity value for the object.
# In CPython, this is commonly the memory address of the object.

x = 5000

print("value:", x)
print("type:", type(x))
print("id:", id(x))
print("hex id:", hex(id(x)))

## 3. Size of Python Objects

`sys.getsizeof()` shows the size of an object in bytes, including CPython object overhead.

This helps us see that a Python integer is not the same as a raw 4-byte C integer.

In [ ]:

# A Python int has object overhead.
# The exact sizes can vary by Python build and operating system.

import sys

values = [
    0,
    1,
    1000,
    10 ** 20,
    10 ** 100,
    3.14,
    True,
    "a",
    "customer",
    [],
    {},
]

for value in values:
    print(repr(value), "=>", type(value).__name__, "=>", sys.getsizeof(value), "bytes")

## 4. Why Python `int` Can Grow Very Large

A C `int` usually has a fixed size, such as 32 bits. A Python `int` can grow as large as memory allows.

CPython stores Python integers using multiple internal chunks. As the integer becomes larger, the object needs more memory.

That is why `10 ** 100` works naturally in Python.

In [ ]:

# As integers become larger, their memory size can increase.

for exponent in [0, 5, 10, 20, 50, 100, 500]:
    number = 10 ** exponent
    print(
        "10 **", exponent,
        "bits:", number.bit_length(),
        "size:", sys.getsizeof(number), "bytes",
    )

## 5. Peeking at Object Header Information with `ctypes`

CPython objects begin with common header fields. We can use `ctypes` to read a simplified view of that header.

This is for learning only. Production Python code should not depend on CPython memory layout.

In [ ]:

# This simplified structure matches the beginning of many CPython objects:
# reference count and type pointer.
# This is CPython-specific, not a general Python language guarantee.

import ctypes

class PyObjectHeader(ctypes.Structure):
    _fields_ = [
        ("ob_refcnt", ctypes.c_ssize_t),
        ("ob_type", ctypes.c_void_p),
    ]

value = 5000
header = PyObjectHeader.from_address(id(value))

print("object value:", value)
print("object address:", hex(id(value)))
print("reference count from ctypes:", header.ob_refcnt)
print("reference count from sys:", sys.getrefcount(value))
print("type pointer:", hex(header.ob_type))
print("id(type(value)):", hex(id(type(value))))

## 6. Why Reference Count Values May Look Different

`sys.getrefcount(value)` usually reports one more reference than expected because passing the object into the function temporarily creates another reference.

Also, Python may keep internal references for optimization. So use reference counts for learning, not for exact business logic.

In [ ]:

# Creating another variable pointing to the same object can increase the reference count.

number = 123456
print("Before alias:", sys.getrefcount(number))

alias = number
print("After alias:", sys.getrefcount(number))

print("number is alias:", number is alias)

## 7. Small Integer Caching

CPython keeps some small integers ready for reuse. This is an implementation optimization.

For example, values such as `-5` through `256` are commonly reused in CPython. Do not write business logic that depends on this behavior.

In [ ]:

# Small integers may be reused, so identity checks can surprise beginners.
# Always use == for numeric equality.

a = 100
b = 100

c = 1000
d = 1000

print("a == b:", a == b)
print("a is b:", a is b)
print("c == d:", c == d)
print("c is d:", c is d)

print("Use == for values. Use is mainly for None checks.")

## 8. Floating Point Internals

Python `float` is usually backed by a C double-precision floating point value.

Most modern systems use IEEE 754 double precision. This means many decimal values cannot be represented exactly in binary.

That is why `0.1 + 0.2` does not produce exactly `0.3`.

In [ ]:

# Floating point values are binary approximations of decimal values.

result = 0.1 + 0.2

print(result)
print(result == 0.3)
print("0.1 as hex:", (0.1).hex())
print("0.2 as hex:", (0.2).hex())
print("result as hex:", result.hex())

## 9. Reading Float Bytes

The `struct` module lets us pack Python values into bytes using C-style binary formats.

This helps us see that a Python `float` object contains object overhead plus an internal floating point payload.

In [ ]:

# struct.pack('d', value) packs a Python float as a C double.
# The bytes shown depend on machine byte order.

import struct

value = 3.14
float_bytes = struct.pack("d", value)

print("value:", value)
print("object size:", sys.getsizeof(value), "bytes")
print("C double payload size:", len(float_bytes), "bytes")
print("payload bytes:", float_bytes)

## 10. Decimal for Exact Decimal Work

For money and exact decimal arithmetic, binary floating point may be the wrong tool.

Python provides `decimal.Decimal`, which is often better for financial calculations and exact decimal representation.

In [ ]:

# Decimal can represent decimal values more naturally than binary float.
# Create Decimal from strings to avoid importing an already-rounded float.

from decimal import Decimal

float_result = 0.1 + 0.2
decimal_result = Decimal("0.1") + Decimal("0.2")

print("float result:", float_result)
print("decimal result:", decimal_result)
print("decimal equals 0.3:", decimal_result == Decimal("0.3"))

## 11. Boolean Is a Subclass of Integer

In Python, `bool` is a subclass of `int`.

`True` behaves like `1`, and `False` behaves like `0` in arithmetic. This is sometimes useful, but it can also surprise people coming from other languages.

In [ ]:

# bool values are objects too, and bool is related to int.

print(isinstance(True, bool))
print(isinstance(True, int))
print(True + True + False)
print(type(True))
print(id(True), id(False))

## 12. Strings Are Unicode Objects

Python 3 strings are Unicode text, not raw bytes.

This matters in data engineering because files, APIs, and databases often involve encodings such as UTF-8.

Use `str` for text and `bytes` for raw binary data.

In [ ]:

# Text and bytes are different types.
# Encoding converts text to bytes. Decoding converts bytes to text.

text = "cafe"
encoded = text.encode("utf-8")
decoded = encoded.decode("utf-8")

print(text, type(text), sys.getsizeof(text))
print(encoded, type(encoded), sys.getsizeof(encoded))
print(decoded, type(decoded))

## 13. Lists Store References, Not Raw Values

A Python list is an object that stores references to other objects.

If a list contains integers, the list does not store raw C integers directly. It stores references to Python integer objects.

In [ ]:

# The list has its own identity.
# Each element is also an object with its own identity.

numbers = [10, 20, 30]

print("list id:", hex(id(numbers)))
print("list size:", sys.getsizeof(numbers), "bytes")

for item in numbers:
    print("item:", item, "type:", type(item).__name__, "id:", hex(id(item)))

## 14. Why This Matters for Data Engineering

Python's object model is powerful and flexible, but it has overhead.

A list of one million Python integers is much heavier than a compact array of one million machine integers.

This is one reason data tools such as NumPy, pandas, PyArrow, and Spark use optimized internal storage instead of plain Python objects for large datasets.

In [ ]:

# This is a small demonstration of list overhead.
# getsizeof(list) does not include the full size of every referenced object.

small_list = list(range(10))

list_only_size = sys.getsizeof(small_list)
items_size = sum(sys.getsizeof(item) for item in small_list)

print("list object only:", list_only_size, "bytes")
print("items total:", items_size, "bytes")
print("rough combined size:", list_only_size + items_size, "bytes")

## 15. Bytecode: Python Operations Are Higher Level

Python code is compiled to bytecode before execution by the CPython virtual machine.

The bytecode operates on Python objects, not directly on raw CPU-level primitive values like C code usually does.

In [ ]:

# dis shows the bytecode generated for a Python function.

import dis

def add_numbers(a, b):
    return a + b

dis.dis(add_numbers)

## 16. Same Operator, Different Object Behavior

The `+` operator does different things depending on the objects involved.

This is possible because Python asks the objects how addition should work for their type.

In [ ]:

# Same operator, different type-specific behavior.

print(10 + 20)
print(1.5 + 2.5)
print("data" + "engineering")
print([1, 2] + [3, 4])

## 17. Where to Read CPython Source Code

If students want to explore CPython internals later, these are useful source files in the CPython project:

- `Include/object.h`: base object structures and macros
- `Include/longobject.h`: integer object declarations
- `Objects/longobject.c`: integer implementation
- `Objects/floatobject.c`: float implementation
- `Objects/listobject.c`: list implementation
- `Objects/dictobject.c`: dictionary implementation
- `Objects/unicodeobject.c`: string implementation

Do not start by reading all of these completely. Pick one simple question, such as "How does Python add two integers?", and trace that path.

## Quick Recap

- In Python 3, `int` is implemented in CPython as `PyLongObject`, not old-style `PyInt`.
- Python values are objects with identity, type information, and object overhead.
- CPython uses reference counting as part of memory management.
- Python integers can grow because they are not fixed-size C integers.
- Python floats are usually backed by C double-precision floating point values.
- Floating point arithmetic can have rounding behavior because decimals are represented in binary.
- Lists store references to objects, not raw primitive values.
- This object model explains both Python's flexibility and some of its performance overhead.